# Appendix D: Stable Curves of Genus Zero

    **Source orientation:** McDuff-Salamon, *J-holomorphic Curves and Symplectic Topology*, Appendix D, printed pp. 619-652; PDF pp. 634-667. Sections: D.1-D.7.

    ## Chapter Goal

    Mobius transformations, cross ratios, trees, labels, splittings, stable curves, Grothendieck-Knudsen space, Gromov topology, cohomology, and examples.

    The guiding question is:

    > How can configurations of marked spheres be compactified by adding tree-shaped degenerations?

    ## Computational Translation Guide

    | Chapter language | Computational object | Inspection target / check |
| --- | --- | --- |
| Mobius action | fractional linear transformation | cross-ratio invariance |
| cross ratio | coordinate on four-point configurations | same after normalization |
| stable curve | tree of spheres with marked special points | vertex stability |
| boundary stratum | partition encoded by an edge | codimension count |
| Gromov topology | convergence of marked domains | tree limit consistency |


## Standalone Reading Guide

    This appendix provides the domain-side compactification used by stable maps. Genus-zero curves are spheres with marked points, and three marked points can be normalized by a Mobius transformation. Four marked points have one complex parameter: the cross ratio. Degenerations occur as that parameter approaches special limiting values, and the compactification records the limit by allowing the sphere to split into a tree of spheres.

Trees are not decorative here; they are coordinates for boundary behavior. A vertex is a component, an edge is a node, and labels indicate which marked points live on which component. Stability again requires at least three special points on every component. Splitting rules describe how the boundary strata fit together and how their cohomology classes can be organized.

The experiment tracks a real cross-ratio parameter approaching a boundary value. It then applies a Mobius transformation and checks that the cross ratio is unchanged. This finite calculation is the domain analogue of the stable-map bookkeeping used throughout the main chapters.

    ## Topics In This Notebook

    - Mobius transformations and normalization of three points
- cross ratios as coordinates for four marked points
- trees, labels, and boundary splittings
- stable curves and the Grothendieck-Knudsen compactification
- examples and cohomological boundary strata

    ## Visualization Storyboard

    - A cross-ratio path shows four marked points degenerating toward a boundary stratum.
- A dependency graph links normalization, trees, stable curves, and compactification.
- A ledger checks cross-ratio invariance under a sample Mobius transformation.


In [ ]:
from pathlib import Path
import csv
import json
import math
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import sympy as sp


def find_book_root(start=None):
    start = (start or Path.cwd()).resolve()
    for base in [start, *start.parents]:
        for candidate in [base, base / "J-Holomorphic-Curves-and-Symplectic-Topology"]:
            if (candidate / "AGENTS.md").exists() and (candidate / "utils").exists():
                return candidate
    raise RuntimeError("JHCST book root not found")


BOOK_ROOT = find_book_root()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import assert_artifact, display_artifact, save_json, save_matplotlib

UNIT = 'appendix-d'
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / UNIT
FIG_DIR = ARTIFACT_ROOT / "figures"
CHECK_DIR = ARTIFACT_ROOT / "checks"
TABLE_DIR = ARTIFACT_ROOT / "tables"
HTML_DIR = ARTIFACT_ROOT / "html"
for folder in [FIG_DIR, CHECK_DIR, TABLE_DIR, HTML_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

BOOK_ROOT


## Proof Visual: Dependency Map

The graph below is a compact proof-state diagram. Read an arrow as "this idea must be under control before the next one can be used." The point is not to replace the analysis with a graph, but to keep the logical dependencies inspectable while the chapter moves between local equations, moduli spaces, compactness, and algebraic counts.


In [ ]:
CONCEPT_NODES = ['Mobius action', 'cross ratio', 'stable curve', 'boundary stratum', 'Gromov topology']
CONCEPT_EDGES = [('Mobius action', 'cross ratio'), ('cross ratio', 'stable curve'), ('stable curve', 'boundary stratum'), ('boundary stratum', 'Gromov topology')]

G = nx.DiGraph()
G.add_nodes_from(CONCEPT_NODES)
G.add_edges_from(CONCEPT_EDGES)
pos = nx.spring_layout(G, seed=37, k=1.35)

fig, ax = plt.subplots(figsize=(9.5, 5.8))
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=18, width=1.8, edge_color="#435466")
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=2300, node_color="#eaf2f8", edgecolors="#1f567d", linewidths=1.5)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=8.5, font_weight="bold")
ax.set_title('Proof dependency map: Stable Curves of Genus Zero')
ax.set_axis_off()
graph_path = save_matplotlib(fig, UNIT, "figures", "proof-dependency-map.png")
plt.close(fig)

graph_check = {
    "nodes": len(CONCEPT_NODES),
    "edges": len(CONCEPT_EDGES),
    "is_directed_acyclic_graph": nx.is_directed_acyclic_graph(G),
    "source_span": '619-652',
    "passed": len(CONCEPT_NODES) >= 5 and nx.is_directed_acyclic_graph(G),
}
graph_json = save_json(graph_check, UNIT, "checks", "proof-dependency-map.json")
display_artifact(graph_path, width=780)
graph_check


## Executable Model

This section builds a small model for one core mechanism in Stable Curves of Genus Zero. The model is intentionally finite and inspectable: it creates an artifact, records a JSON check, and gives a learner a parameter to perturb in the applied lab.


In [ ]:
def cross_ratio(a, b, c, d):
    return ((a - c) * (b - d)) / ((a - d) * (b - c))

lam = np.geomspace(1.0, 0.03, 80)
cr_values = []
for value in lam:
    # points are value, 1, a large proxy for infinity, and 0
    cr_values.append(float(value))

def mobius(z):
    return (2 * z + 1) / (z + 3)

sample = 0.37
original = cross_ratio(sample, 1.0, 1e9, 0.0)
transformed = cross_ratio(mobius(sample), mobius(1.0), mobius(1e9), mobius(0.0))

fig, ax = plt.subplots(figsize=(7.0, 4.8))
ax.plot(range(len(lam)), cr_values, color="#756bb1")
ax.set_yscale("log")
ax.set_xlabel("degeneration step")
ax.set_ylabel("cross-ratio parameter")
ax.set_title("Marked sphere degeneration toward a boundary stratum")
ax.grid(True, which="both", alpha=0.25)
fig_path = save_matplotlib(fig, UNIT, "figures", "cross-ratio-boundary-degeneration.png")
plt.close(fig)

check = {
    "sample_cross_ratio": float(original),
    "mobius_transformed_cross_ratio": float(transformed),
    "absolute_difference": float(abs(original - transformed)),
    "limit_toward_boundary": float(cr_values[-1]),
    "passed": bool(abs(original - transformed) < 1e-6 and cr_values[-1] < 0.05),
}
check_path = save_json(check, UNIT, "checks", "cross-ratio-invariance-checks.json")
display_artifact(fig_path, width=680)
check


## Invariant Ledger

The ledger records the chapter vocabulary as computational objects plus explicit checks. It is a small source map inside the notebook: every row names what should be inspected when the figure or experiment is changed.


In [ ]:
ledger_rows = [{'item': 'Mobius action', 'computational_object': 'fractional linear transformation', 'check': 'cross-ratio invariance'}, {'item': 'cross ratio', 'computational_object': 'coordinate on four-point configurations', 'check': 'same after normalization'}, {'item': 'stable curve', 'computational_object': 'tree of spheres with marked special points', 'check': 'vertex stability'}, {'item': 'boundary stratum', 'computational_object': 'partition encoded by an edge', 'check': 'codimension count'}, {'item': 'Gromov topology', 'computational_object': 'convergence of marked domains', 'check': 'tree limit consistency'}]
table_path = TABLE_DIR / "invariant-ledger.csv"
with table_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=["item", "computational_object", "check"])
    writer.writeheader()
    writer.writerows(ledger_rows)

ledger_check = {
    "row_count": len(ledger_rows),
    "items": [row["item"] for row in ledger_rows],
    "has_source_specific_checks": all(row["check"] for row in ledger_rows),
    "passed": len(ledger_rows) >= 5 and all(row["check"] for row in ledger_rows),
}
ledger_json = save_json(ledger_check, UNIT, "checks", "invariant-ledger.json")
display_artifact(table_path)
ledger_check


## Applied Lab

Move the fourth point toward 1 instead of 0. The same cross-ratio machinery detects a different boundary partition.

The intended workflow is to change one parameter, rerun the executable model, and then inspect both the figure and JSON check. If the visual impression and the invariant check disagree, trust the check first and then ask what the visualization is hiding.


## Takeaways

    - Cross ratios are local coordinates for marked four-pointed spheres.
- Stable genus-zero curves compactify configurations by adding tree-shaped nodal domains.
- The same tree language organizes both domains and stable maps.

    ## Sanity Checks

    The final cell asserts that the generated figures, ledgers, and JSON checks exist, are nonempty, and report successful chapter-specific invariants.


In [ ]:
expected = [
    FIG_DIR / "proof-dependency-map.png",
    FIG_DIR / 'cross-ratio-boundary-degeneration.png',
    CHECK_DIR / "proof-dependency-map.json",
    CHECK_DIR / 'cross-ratio-invariance-checks.json',
    CHECK_DIR / "invariant-ledger.json",
    TABLE_DIR / "invariant-ledger.csv",
]
for path in expected:
    min_bytes = 80 if path.suffix == ".csv" else 512
    assert_artifact(path, min_bytes=min_bytes)

for path in [CHECK_DIR / "proof-dependency-map.json", CHECK_DIR / 'cross-ratio-invariance-checks.json', CHECK_DIR / "invariant-ledger.json"]:
    data = json.loads(path.read_text(encoding="utf-8"))
    assert data.get("passed") is True, path

print(f"Validated {len(expected)} artifacts for {UNIT}")
